# 04 Interaction models

Panel regressions and the drought-cold interaction.

| Output | Manuscript |
| --- | --- |
| `figures/figure_08_drought_cold_heatmap.pdf` | Figure 8 |
| `figures/figure_S02_leave_one_year_out.pdf` | Figure S2 |
| printed tables | Tables S1, S2, S4 |

Reads only `data/processed/`. No raw file is opened.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from src import data_loading as dl

dl.set_plot_style()
dl.describe_paths()

panel = dl.load_mortality_panel(zscore_reference="study")

print(f"panel: {len(panel)} rows, {panel['Aimag'].nunique()} aimags, "
      f"{panel['Year'].min()}-{panel['Year'].max()}")
print(f"winter Z-score reference period: {dl.ZSCORE_REFERENCE[0]}-{dl.ZSCORE_REFERENCE[1]}")

## Table S1. Previous-summer SPEI-3 and annual mortality

Three specifications on the unweighted species-mean rate `All`: pooled OLS, aimag
fixed effects, and aimag plus year fixed effects with standard errors clustered by
aimag. Coefficients are reported in percentage points.

In [ ]:
model_data = panel[["Aimag", "Year", "All", "All_SFU", "SPEI_PrevSummer"]].dropna(
    subset=["All", "SPEI_PrevSummer"]).copy()
model_data["All_pct"] = model_data["All"] * 100
model_data["All_SFU_pct"] = model_data["All_SFU"] * 100

pooled = smf.ols("All_pct ~ SPEI_PrevSummer", data=model_data).fit()
aimag_fe = smf.ols("All_pct ~ SPEI_PrevSummer + C(Aimag)", data=model_data).fit()
two_way_fe = smf.ols("All_pct ~ SPEI_PrevSummer + C(Aimag) + C(Year)",
                     data=model_data).fit()
two_way_fe_clustered = two_way_fe.get_robustcov_results(
    cov_type="cluster", groups=model_data["Aimag"])


def coefficient_row(name, result, variable="SPEI_PrevSummer"):
    names = list(result.model.exog_names)
    index = names.index(variable)
    params = np.asarray(result.params)
    intervals = np.asarray(result.conf_int(alpha=0.05))
    pvalues = np.asarray(result.pvalues)
    return {"Model": name,
            "SPEI coefficient": params[index],
            "CI lower": intervals[index, 0],
            "CI upper": intervals[index, 1],
            "p-value": pvalues[index],
            "R2": result.rsquared,
            "n": int(result.nobs)}


table_s1 = pd.DataFrame([
    coefficient_row("Pooled OLS", pooled),
    coefficient_row("Aimag fixed effects", aimag_fe),
    coefficient_row("Aimag and year fixed effects", two_way_fe_clustered),
])

print("Table S1. Percentage-point change in annual mortality per one-unit increase "
      "in mean June-August SPEI-3")
print(table_s1.round(4).to_string(index=False))
print("\nAs proportions: "
      + ", ".join(f"{v / 100:.5f}" for v in table_s1["SPEI coefficient"]))

In [ ]:
sfu_two_way = smf.ols("All_SFU_pct ~ SPEI_PrevSummer + C(Aimag) + C(Year)",
                      data=model_data).fit()
sfu_two_way_clustered = sfu_two_way.get_robustcov_results(
    cov_type="cluster", groups=model_data["Aimag"])

print("Same specification on the Equation (2) SFU-weighted rate")
print(pd.DataFrame([coefficient_row("Aimag and year fixed effects, All_SFU",
                                    sfu_two_way_clustered)]).round(4).to_string(index=False))

## Figure 8. Mortality by previous-summer moisture and winter severity

Rows bin previous-summer SPEI-3 at -1 and -0.5. Columns bin the within-aimag DJF
temperature Z-score at -1 and 0. Cell values are mean SFU-weighted mortality.

In [ ]:
heatmap_data = panel.dropna(
    subset=["SPEI_PrevSummer", "Temp_Winter_Z", "All_SFU"]).copy()

SPEI_LABELS = ["Severe drought\n(SPEI < \u22121)",
               "Mild drought\n(\u22121 \u2264 SPEI < \u22120.5)",
               "Normal / wet\n(SPEI \u2265 \u22120.5)"]
TEMP_LABELS = ["Severe cold\n(Z < \u22121)",
               "Moderate cold\n(\u22121 \u2264 Z < 0)",
               "Mild winter\n(Z \u2265 0)"]

heatmap_data["spei_class"] = pd.cut(heatmap_data["SPEI_PrevSummer"],
                                    bins=dl.SPEI_EDGES, labels=False,
                                    right=False, include_lowest=True)
heatmap_data["temp_class"] = pd.cut(heatmap_data["Temp_Winter_Z"],
                                    bins=dl.TEMP_EDGES, labels=False,
                                    right=False, include_lowest=True)

grid = np.full((len(SPEI_LABELS), len(TEMP_LABELS)), np.nan)
counts = np.zeros_like(grid, dtype=int)

for (row_index, column_index), group in heatmap_data.groupby(
        ["spei_class", "temp_class"], observed=True):
    grid[int(row_index), int(column_index)] = group["All_SFU"].mean() * 100
    counts[int(row_index), int(column_index)] = len(group)

if np.isnan(grid).all():
    raise ValueError("No complete observations are available for the heatmap.")

table_8 = pd.DataFrame(grid, index=[s.replace("\n", " ") for s in SPEI_LABELS],
                       columns=[t.replace("\n", " ") for t in TEMP_LABELS])
print("Mean SFU-weighted mortality (%) by category")
print(table_8.round(2).to_string())
print("\nObservations per category")
print(pd.DataFrame(counts, index=table_8.index,
                   columns=table_8.columns).to_string())

In [ ]:
HEATMAP_SCALE = 3.0

with mpl.rc_context({"font.size": 8 * HEATMAP_SCALE,
                     "axes.labelsize": 9 * HEATMAP_SCALE,
                     "xtick.labelsize": 8 * HEATMAP_SCALE,
                     "ytick.labelsize": 8 * HEATMAP_SCALE,
                     "pdf.fonttype": 42, "ps.fonttype": 42}):

    fig, ax = plt.subplots(
        figsize=(dl.AGU_MAX_WIDTH_IN * HEATMAP_SCALE, 4.2 * HEATMAP_SCALE),
        facecolor="white")
    ax.set_facecolor("white")

    vmax = float(np.nanmax(grid))
    image = ax.imshow(grid, cmap="YlOrRd", aspect="auto", vmin=0, vmax=vmax,
                      interpolation="none")

    colorbar = fig.colorbar(image, ax=ax, shrink=0.88, pad=0.025, aspect=28)
    colorbar.set_label("Mean aimag-year SFU-weighted mortality (%)",
                       fontsize=8.5 * HEATMAP_SCALE, labelpad=8 * HEATMAP_SCALE)
    colorbar.ax.tick_params(labelsize=8 * HEATMAP_SCALE, width=0.45 * HEATMAP_SCALE,
                            length=2.4 * HEATMAP_SCALE, pad=2.4 * HEATMAP_SCALE)

    ax.set_xticks(np.arange(len(TEMP_LABELS)), TEMP_LABELS,
                  fontsize=8 * HEATMAP_SCALE, linespacing=1.15)
    ax.set_yticks(np.arange(len(SPEI_LABELS)), SPEI_LABELS,
                  fontsize=8 * HEATMAP_SCALE, linespacing=1.15)
    ax.tick_params(axis="x", which="major", length=0, pad=6 * HEATMAP_SCALE)
    ax.tick_params(axis="y", which="major", length=0, pad=5.5 * HEATMAP_SCALE)

    ax.set_xlabel("DJF severity in year $t$ "
                  "(ERA5 temperature Z-score within aimag, "
                  f"{dl.ZSCORE_REFERENCE[0]}\u2013{dl.ZSCORE_REFERENCE[1]} reference)",
                  fontsize=9 * HEATMAP_SCALE, labelpad=9 * HEATMAP_SCALE)
    ax.set_ylabel("Previous-summer moisture\n(JJA SPEI-3 in year $t-1$)",
                  fontsize=9 * HEATMAP_SCALE, labelpad=9 * HEATMAP_SCALE)

    for row_index in range(grid.shape[0]):
        for column_index in range(grid.shape[1]):
            value = grid[row_index, column_index]
            if np.isnan(value):
                continue
            text_color = "white" if value > vmax * 0.58 else "#111111"
            ax.text(column_index, row_index - 0.13, f"{value:.1f}%", ha="center",
                    va="center", fontsize=9 * HEATMAP_SCALE, fontweight="bold",
                    color=text_color, zorder=5)
            ax.text(column_index, row_index + 0.26,
                    f"$n={counts[row_index, column_index]}$", ha="center",
                    va="center", fontsize=8 * HEATMAP_SCALE, color=text_color,
                    alpha=0.95, zorder=5)

    ax.set_xticks(np.arange(-0.5, grid.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, grid.shape[0], 1), minor=True)
    ax.grid(which="minor", color="white", linestyle="-",
            linewidth=1.0 * HEATMAP_SCALE)
    ax.grid(which="major", visible=False)
    ax.tick_params(which="minor", bottom=False, left=False)

    for spine in ax.spines.values():
        spine.set_visible(False)

    fig.subplots_adjust(left=0.25, right=0.91, top=0.82, bottom=0.27)

dl.save_figure(fig, "figure_08_drought_cold_heatmap", scale=HEATMAP_SCALE)
plt.show()

## Table S2. Continuous interaction model

Aimag and year fixed effects, standard errors two-way clustered by aimag and year.

In [ ]:
interaction_data = panel[["Aimag", "Year", "All_SFU", "SPEI_PrevSummer",
                          "Temp_Winter_Z"]].replace(
    [np.inf, -np.inf], np.nan).dropna().copy()
interaction_data["All_SFU_pct"] = interaction_data["All_SFU"] * 100

FORMULA = "All_SFU_pct ~ SPEI_PrevSummer * Temp_Winter_Z + C(Aimag) + C(Year)"
INTERACTION_TERM = "SPEI_PrevSummer:Temp_Winter_Z"


def fit_two_way_clustered(data):
    groups = np.column_stack([pd.factorize(data["Aimag"])[0],
                              pd.factorize(data["Year"])[0]])
    return smf.ols(FORMULA, data=data).fit(
        cov_type="cluster", cov_kwds={"groups": groups}, use_t=True)


interaction_model = fit_two_way_clustered(interaction_data)
intervals = interaction_model.conf_int()

table_s2 = pd.DataFrame([
    {"Predictor": term,
     "Coefficient": interaction_model.params[term],
     "CI lower": intervals.loc[term, 0],
     "CI upper": intervals.loc[term, 1],
     "p-value": interaction_model.pvalues[term]}
    for term in ["SPEI_PrevSummer", "Temp_Winter_Z", INTERACTION_TERM]])

beta3 = interaction_model.params[INTERACTION_TERM]

print(f"Sample: {len(interaction_data)} aimag-year observations, "
      f"{interaction_data['Aimag'].nunique()} aimags, "
      f"{interaction_data['Year'].nunique()} years")
print("\nTable S2. Coefficients in percentage points")
print(table_s2.round(4).to_string(index=False))

## Table S4 and Figure S2. Leave-one-year-out test

The model is refitted 33 times, each time excluding all observations from one year.

In [ ]:
leave_one_out_rows = []
for omitted_year in sorted(interaction_data["Year"].unique()):
    reduced = interaction_data[interaction_data["Year"] != omitted_year]
    try:
        result = fit_two_way_clustered(reduced)
    except Exception as error:
        print(f"Model failed when omitting {omitted_year}: {error}")
        continue
    bounds = result.conf_int().loc[INTERACTION_TERM]
    leave_one_out_rows.append({
        "Omitted year": int(omitted_year),
        "Interaction coefficient": result.params[INTERACTION_TERM],
        "CI lower": bounds.iloc[0],
        "CI upper": bounds.iloc[1],
        "p-value": result.pvalues[INTERACTION_TERM]})

leave_one_out = pd.DataFrame(leave_one_out_rows)
leave_one_out["Change from full model"] = (
    leave_one_out["Interaction coefficient"] - beta3)

positive = (leave_one_out["Interaction coefficient"] > 0).sum()
significant = ((leave_one_out["Interaction coefficient"] > 0)
               & (leave_one_out["p-value"] < 0.05)).sum()

print(f"Full-model interaction estimate {beta3:.4f} percentage points")
print(f"Leave-one-year-out range {leave_one_out['Interaction coefficient'].min():.4f} "
      f"to {leave_one_out['Interaction coefficient'].max():.4f}")
print(f"{positive} of {len(leave_one_out)} estimates positive; "
      f"{significant} of {len(leave_one_out)} with p < 0.05")

table_s4 = leave_one_out.reindex(
    leave_one_out["Change from full model"].abs()
    .sort_values(ascending=False).index).head(10)
print("\nTable S4. Ten omissions producing the largest change")
print(table_s4.round(4).to_string(index=False))

In [ ]:
LOYO_SCALE = 2.2

with mpl.rc_context({"font.size": 8 * LOYO_SCALE,
                     "axes.labelsize": 9 * LOYO_SCALE,
                     "xtick.labelsize": 8 * LOYO_SCALE,
                     "ytick.labelsize": 8 * LOYO_SCALE,
                     "legend.fontsize": 8 * LOYO_SCALE,
                     "pdf.fonttype": 42, "ps.fonttype": 42}):

    fig, ax = plt.subplots(
        figsize=(dl.AGU_MAX_WIDTH_IN * 0.75 * LOYO_SCALE, 5.2 * LOYO_SCALE),
        facecolor="white")

    highlight_years = {2001, 2010}
    for _, row in leave_one_out.iterrows():
        year = int(row["Omitted year"])
        color = dl.DRY_COLOR if year in highlight_years else "#4C72B0"
        ax.plot([row["CI lower"], row["CI upper"]], [year, year], color=color,
                linewidth=0.6 * LOYO_SCALE, alpha=0.85)
        ax.scatter(row["Interaction coefficient"], year, color=color,
                   s=6 * LOYO_SCALE ** 2, zorder=3)

    ax.axvline(beta3, color="black", linestyle="--", linewidth=0.6 * LOYO_SCALE,
               label="Full-model estimate")
    ax.axvline(0, color="gray", linewidth=0.45 * LOYO_SCALE)
    ax.set_yticks(leave_one_out["Omitted year"])
    ax.invert_yaxis()
    ax.set_xlabel("Interaction coefficient (percentage points)",
                  fontsize=9 * LOYO_SCALE)
    ax.grid(axis="x", linestyle="--", alpha=0.30)
    ax.grid(axis="y", visible=False)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=8 * LOYO_SCALE)

dl.save_figure(fig, "figure_S02_leave_one_year_out", scale=LOYO_SCALE)
plt.show()

## Sensitivity to the winter Z-score reference period

The submitted analysis standardises DJF temperature within aimag over 1992-2024.
Because Mongolian winters warmed over the ERA5 record, a 1950-2024 reference shifts
the cold-anomaly bins. Both fits are reported so the Methods can state which is used.

In [ ]:
for reference in ("study", "climate"):
    alternative = dl.load_mortality_panel(zscore_reference=reference)
    alternative_data = alternative[["Aimag", "Year", "All_SFU", "SPEI_PrevSummer",
                                    "Temp_Winter_Z"]].replace(
        [np.inf, -np.inf], np.nan).dropna().copy()
    alternative_data["All_SFU_pct"] = alternative_data["All_SFU"] * 100
    fitted = fit_two_way_clustered(alternative_data)
    severe_cold = (alternative_data["Temp_Winter_Z"]
                   < dl.SEVERE_COLD_THRESHOLD).sum()
    print(f"{reference:8s} reference  "
          f"interaction {fitted.params[INTERACTION_TERM]:+.4f} pp  "
          f"p = {fitted.pvalues[INTERACTION_TERM]:.4f}  "
          f"severe-cold aimag-years {severe_cold}")